# Lab 6 — Agentes sísmicos (Agno + Ollama) — Vía IA-asistida

**Sesión:** 10 — agentes de IA y orquestación ML

**Público:** ingenieros civiles **sin experiencia en programación** · **Entorno:** GitHub Codespaces o `labs/.venv`

## Cómo trabajar (Copilot, Gemini, Cursor, etc.)

1. Abre este repo en **Codespaces** o activa `source labs/.venv/bin/activate`.
2. **Ejecuta** las celdas pre-escritas (carga de datos, gráficos base) en orden.
3. Lee la **guía IA** de la sección: objetivo, qué considerar y el prompt sugerido.
4. Copia el prompt a tu asistente; pega el código generado en la celda **«Aquí coloca tu solución»**.
5. Ejecuta tu celda y la **Autoevaluación**; busca ✅ antes de avanzar.
6. Registra tus prompts en [`prompts_entregados.md`](prompts_entregados.md) (entrega obligatoria).
7. Referencia docente: `agentes_estructuras_solucion.ipynb` (no distribuir al inicio).

**La IA propone; tú validas** con autoevaluación, gráficos y sentido físico/estructural.

LLM: **Ollama** (`llama3.2:3b`). Setup: `bash labs/lab6/_ollama_setup.sh`


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import requests

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    TELEMETRIA_DB, RUTA_CSV, RUTA_PKL,
    evaluar_drift_asce7, predecir_probabilidad_dano, generar_latex_report,
    leer_meta_modelo, cargar_modelo_sismico,
    verificar_pasos_react, verificar_entorno, verificar_ollama,
    verificar_telemetria_b, verificar_resultado_norma_b, verificar_ml_b,
    verificar_agente_config, verificar_respuesta_agente,
    verificar_tabla_comparativa, verificar_latex_report, verificar, resumen_seccion,
)

MODELO_LLM = 'llama3.2:3b'
OLLAMA_URL = 'http://localhost:11434'
TRAZA_HERRAMIENTAS: list[str] = []

print('✅ Entorno listo — LLM: Ollama', MODELO_LLM, '@', OLLAMA_URL)
print('   Edificios demo:', list(TELEMETRIA_DB.keys()))


## Caja de herramientas del Lab 6

> Construir un **agente inspector** que coordina telemetría, norma ASCE 7, un **MLP pre-entrenado** y exportación LaTeX — el **LLM (Ollama) no inventa cálculos**, solo orquesta tools.

| Herramienta | Rol | Analogía en obra |
|-------------|-----|------------------|
| **Ollama** | LLM local (`llama3.2:3b`) — razonamiento y dictamen | Ingeniero senior que redacta informes |
| **Agno** | Bucle ReAct + tool calling | Coordinador de oficina técnica |
| **MLP (.pkl)** | Red neuronal entrenada offline | Modelo especialista del laboratorio |
| **joblib** | Carga del best model | Archivo `.pkl` listo para producción |
| **LaTeX** | Plantilla `informe_sismico.tex` | Memoria de inspección trazable |

**Flujo:** Usuario → Agente (Ollama) → {telemetría, norma, MLP, LaTeX} → informe `.tex`


In [ ]:
# --- PRE-ESCRITO: conoce tu best model (no re-entrenar) ---
import subprocess
subprocess.run([sys.executable, '_preparar_datos.py'], check=False)

meta = leer_meta_modelo()
print('Tipo:', meta.get('tipo'), '| Arquitectura:', meta.get('architecture'))
print('test_r2:', meta.get('test_r2'), '| Features:', meta.get('features'))
_ = cargar_modelo_sismico()
print('✅ Best model cargado desde', RUTA_PKL)


## Dataset — Escenarios sísmico-estructurales

| ID demo | Narrativa |
|---------|-----------|
| **BLDG-A** | Acero / suelo duro — norma OK + ML bajo |
| **BLDG-B** | Concreto / suelo blando — **falla norma** + alerta ML |
| **BLDG-C** | Composite — **norma OK** pero **ML crítico** |

Detalle: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Chat vs agente (ReAct)

### 📘 Subpreguntas
- **1.a** ¿Por qué un LLM solo no puede abrir tu base de datos de sensores?
- **1.b** ¿Qué aporta el bucle ReAct frente a un prompt único?


#### ✍️ Tu respuesta (opcional, 2–3 líneas)




### 🤖 Guía IA — Sección 1: Bucle ReAct

**Objetivo:** Definir los pasos del bucle ReAct (Razonamiento + Acción) que seguirá el agente.

**Tu código debe lograr**
- Definir `PASOS_REACT` como lista con al menos 4 pasos
- Incluir observación, razonamiento, acción con tools y verificación
- Imprimir la lista numerada

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `PASOS_REACT`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- ReAct = el LLM decide qué tool llamar según el objetivo
- En obra: el agente no calcula drift — invoca herramientas acotadas

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Estoy en Lab 6 (agentes sísmicos). Genera código que:
1) defina PASOS_REACT = ["Observar objetivo del usuario", "Razonar siguiente paso", "Invocar herramienta (tool call)", "Verificar resultado y sintetizar"]
2) imprima "Bucle ReAct:" y enumere cada paso numerado
No uses imports nuevos.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · PASOS_REACT

# Tu código debe:
#   1. Lista ≥4 pasos ReAct
#   2. Imprimir numerada



In [ ]:
# --- Autoevaluación 1 ---
# Requiere (celda «Aquí coloca tu solución»): `PASOS_REACT`
r = []
try:
    r = verificar_pasos_react(PASOS_REACT)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('1 — ReAct', r)


## Pregunta 2 — Ollama + best model MLP

### 📘 Subpreguntas
- **2.a** ¿Ventaja de Ollama en localhost:11434?
- **2.b** ¿Por qué no re-entrenamos el MLP en clase?


In [ ]:
# --- PRE-ESCRITO: diagnóstico Ollama ---
# Si OLLAMA_OK es False: bash labs/lab6/_ollama_setup.sh
print("Comprueba Ollama antes de las secciones 6–7.")


### 🤖 Guía IA — Sección 2: Entorno Ollama + best model

**Objetivo:** Verificar Ollama, el CSV sísmico y el best model MLP entregado (.pkl).

**Tu código debe lograr**
- Definir `MODELO_LLM = 'llama3.2:3b'` y comprobar `/api/tags` → `OLLAMA_OK`
- Comprobar que existe `data/earthquake_risk_model.pkl` → `PKL_OK`
- Contar filas del CSV → `N_FILAS_CSV`
- Leer meta → `TIPO_MODELO` debe contener 'MLP'

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `MODELO_LLM`, `OLLAMA_OK`, `PKL_OK`, `N_FILAS_CSV`, `TIPO_MODELO`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Si OLLAMA_OK es False: bash labs/lab6/_ollama_setup.sh
- No re-entrenes el modelo — solo verifica el .pkl entregado

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter Lab 6 necesito verificar entorno.
Genera código que:
1) MODELO_LLM = "llama3.2:3b"
2) OLLAMA_URL = "http://localhost:11434"; intente requests.get(.../api/tags); OLLAMA_OK = status==200
3) PKL_OK = Path("data/earthquake_risk_model.pkl").is_file()
4) N_FILAS_CSV = len(pd.read_csv("data/seismic_data.csv"))
5) meta = leer_meta_modelo(); TIPO_MODELO = meta.get("tipo","?")
6) imprima OLLAMA_OK, PKL_OK, N_FILAS_CSV, TIPO_MODELO
Usa requests, pandas, Path, leer_meta_modelo de _verificar.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · MODELO_LLM
#   · OLLAMA_OK
#   · PKL_OK
#   · N_FILAS_CSV
#   · TIPO_MODELO

# Tu código debe:
#   1. Verificar Ollama
#   2. Verificar pkl y CSV
#   3. Leer TIPO_MODELO



In [ ]:
# --- Autoevaluación 2 ---
# Requiere (celda «Aquí coloca tu solución»): `MODELO_LLM`, `OLLAMA_OK`, `PKL_OK`, `N_FILAS_CSV`, `TIPO_MODELO`
r = []
try:
    r = verificar_entorno(OLLAMA_OK, PKL_OK, N_FILAS_CSV, TIPO_MODELO)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('2 — Ollama + MLP', r)


## Pregunta 3 — Tool: telemetría

### 📘 Subpreguntas
- **3.a** ¿Qué campos necesitas para la norma y el ML?


In [ ]:
# --- PRE-ESCRITO: edificios disponibles ---
print(list(TELEMETRIA_DB.keys()))


### 🤖 Guía IA — Sección 3: Tool telemetría

**Objetivo:** Implementar `get_building_telemetry(building_id)` usando `TELEMETRIA_DB`.

**Tu código debe lograr**
- Función retorna dict del edificio o error si no existe
- Probar con BLDG-B y guardar en `TELEMETRIA_B`
- Imprimir claves principales

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `TELEMETRIA_B`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- TELEMETRIA_DB ya está en _verificar — no inventes otra base
- BLDG-B es concreto en suelo blando con drift > 2%

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Lab 6 — implementa get_building_telemetry(building_id: str) -> dict
Usa TELEMETRIA_DB importado de _verificar.
Luego TELEMETRIA_B = get_building_telemetry("BLDG-B") e imprime name, max_drift_ratio, material.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · TELEMETRIA_B

# Tu código debe:
#   1. Definir get_building_telemetry
#   2. Asignar TELEMETRIA_B



In [ ]:
# --- Autoevaluación 3 ---
# Requiere (celda «Aquí coloca tu solución»): `TELEMETRIA_B`
r = []
try:
    r = verificar_telemetria_b(TELEMETRIA_B)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('3 — Telemetría', r)


## Pregunta 4 — Tool: norma ASCE 7

### 📘 Subpreguntas
- **4.a** ¿Por qué BLDG-B falla si el límite de concreto es 2%?


### 🤖 Guía IA — Sección 4: Tool normativa ASCE 7

**Objetivo:** Implementar `check_seismic_code_compliance(max_drift_ratio, material)` con evaluar_drift_asce7.

**Tu código debe lograr**
- Usar evaluar_drift_asce7 de _verificar
- Aplicar a BLDG-B → `RESULTADO_NORMA_B`
- Imprimir resultado

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `RESULTADO_NORMA_B`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Límite depende del material: acero 2.5%, concreto 2.0%
- BLDG-B debe arrojar FALLO

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Implementa check_seismic_code_compliance(max_drift_ratio, material) -> str
usando evaluar_drift_asce7 de _verificar.
RESULTADO_NORMA_B = check_seismic_code_compliance(TELEMETRIA_B["max_drift_ratio"], TELEMETRIA_B["material"])
print(RESULTADO_NORMA_B)
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · RESULTADO_NORMA_B

# Tu código debe:
#   1. Definir check_seismic_code_compliance
#   2. Evaluar BLDG-B



In [ ]:
# --- Autoevaluación 4 ---
# Requiere (celda «Aquí coloca tu solución»): `RESULTADO_NORMA_B`
r = []
try:
    r = verificar_resultado_norma_b(RESULTADO_NORMA_B)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('4 — Norma', r)


## Pregunta 5 — Tool: MLP (.pkl)

### 📘 Subpreguntas
- **5.a** ¿Diferencia entre fallo normativo y alerta ML?


### 🤖 Guía IA — Sección 5: Tool ML (best model .pkl)

**Objetivo:** Implementar predict_earthquake_vulnerability con predecir_probabilidad_dano.

**Tu código debe lograr**
- Usar predecir_probabilidad_dano con pga, suelo, periodo, drift, height, material
- Asignar PROB_RIESGO_B y MSG_ML_B
- Imprimir probabilidad y mensaje

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `PROB_RIESGO_B`, `MSG_ML_B`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- El .pkl es un MLPRegressor — no re-entrenar
- PROB_RIESGO_B debe ser > 0.70 para BLDG-B

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Implementa predict_earthquake_vulnerability(...) -> str que llame predecir_probabilidad_dano
con todos los campos de TELEMETRIA_B (expected_pga_g, soil_type_index, structural_period_s,
max_drift_ratio, height_m, material). Asigna PROB_RIESGO_B (float) y MSG_ML_B (str). Imprime ambos.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · PROB_RIESGO_B
#   · MSG_ML_B

# Tu código debe:
#   1. Definir predict_earthquake_vulnerability
#   2. Calcular PROB_RIESGO_B y MSG_ML_B



In [ ]:
# --- Autoevaluación 5 ---
# Requiere (celda «Aquí coloca tu solución»): `PROB_RIESGO_B`, `MSG_ML_B`
r = []
try:
    r = verificar_ml_b(PROB_RIESGO_B, MSG_ML_B)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('5 — MLP', r)


## Pregunta 6 — Agente Agno + Ollama

### 📘 Subpreguntas
- **6.a** ¿Por qué pasamos funciones Python como `tools`?
- **6.b** ¿Qué hace `show_tool_calls=True`?


In [ ]:
# --- PRE-ESCRITO: import Agno + Ollama ---
from agno.agent import Agent
from agno.models.ollama import Ollama
print("Agno listo — modelo Ollama:", MODELO_LLM)


### 🤖 Guía IA — Sección 6: Configurar agente Agno

**Objetivo:** Crear structural_bot con Ollama y 4 tools (telemetría, norma, ML, LaTeX).

**Tu código debe lograr**
- Importar Agent y Ollama de agno
- Pasar las 4 funciones como tools
- instructions con flujo: telemetría → norma → ML → dictamen → export LaTeX
- N_TOOLS = 4

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `N_TOOLS`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- model=Ollama(id=MODELO_LLM)
- show_tool_calls=True para ver ReAct en consola

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Crea structural_bot = Agent(model=Ollama(id=MODELO_LLM), tools=[get_building_telemetry,
check_seismic_code_compliance, predict_earthquake_vulnerability, export_latex_report],
instructions=[...5-6 instrucciones en español...], show_tool_calls=True, markdown=True)
N_TOOLS = 4
No ejecutes aún el agente.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · N_TOOLS

# Tu código debe:
#   1. Definir export_latex_report si falta
#   2. Crear structural_bot
#   3. N_TOOLS=4



In [ ]:
# --- Autoevaluación 6 ---
# Requiere (celda «Aquí coloca tu solución»): `N_TOOLS`
r = []
try:
    r = verificar_agente_config(structural_bot, N_TOOLS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('6 — Agente', r)


## Pregunta 7 — Ejecutar agente (BLDG-B)

### 📘 Subpreguntas
- **7.a** ¿Qué tool llamó primero el agente?
- **7.b** ¿Cómo encadenó salidas entre tools?


### 🤖 Guía IA — Sección 7: Ejecutar agente BLDG-B

**Objetivo:** Ejecutar el agente sobre BLDG-B y capturar RESPUESTA_B y TRAZA_HERRAMIENTAS.

**Tu código debe lograr**
- Consulta en lenguaje natural sobre inspección BLDG-B
- Guardar texto final en RESPUESTA_B
- TRAZA_HERRAMIENTAS = lista con pasos registrados (usa lista global o atributo del run)

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `RESPUESTA_B`, `TRAZA_HERRAMIENTAS`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Requiere OLLAMA_OK=True
- Si falla Ollama, simula traza mínima documentando el error

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Ejecuta structural_bot.run(...) o print_response con consulta de inspección BLDG-B.
Captura RESPUESTA_B (str) y TRAZA_HERRAMIENTAS (list[str] con nombres de tools invocadas).
Si OLLAMA_OK es False, RESPUESTA_B = mensaje de error y TRAZA_HERRAMIENTAS = [].
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · RESPUESTA_B
#   · TRAZA_HERRAMIENTAS

# Tu código debe:
#   1. Ejecutar agente
#   2. Capturar respuesta y traza



In [ ]:
# --- Autoevaluación 7 ---
# Requiere (celda «Aquí coloca tu solución»): `RESPUESTA_B`, `TRAZA_HERRAMIENTAS`
r = []
try:
    r = verificar_respuesta_agente(RESPUESTA_B, TRAZA_HERRAMIENTAS) if OLLAMA_OK else [True]
    if not OLLAMA_OK: print('⚠️ Autoevaluación 7 omitida (sin Ollama).')
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('7 — Ejecución', r)


## Pregunta 8 — Contraste A / B / C

### 📘 Subpreguntas
- **8.a** ¿Por qué C pasa norma pero ML alerta?


### 🤖 Guía IA — Sección 8: Contraste A / B / C

**Objetivo:** Tabla comparativa determinista vs probabilístico para los 3 edificios.

**Tu código debe lograr**
- Para BLDG-A, B, C ejecutar tools de norma y ML (sin agente)
- Construir TABLA_COMPARATIVA DataFrame con columnas: edificio, drift, norma, prob_ml

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `TABLA_COMPARATIVA`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- BLDG-C: norma OK pero ML alto — clave pedagógica

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Crea TABLA_COMPARATIVA (pandas DataFrame) con filas BLDG-A, BLDG-B, BLDG-C.
Columnas: edificio, drift, resultado_norma, prob_ml.
Usa get_building_telemetry, check_seismic_code_compliance y predecir_probabilidad_dano.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · TABLA_COMPARATIVA

# Tu código debe:
#   1. Iterar A/B/C
#   2. Armar DataFrame comparativo



In [ ]:
# --- Autoevaluación 8 ---
# Requiere (celda «Aquí coloca tu solución»): `TABLA_COMPARATIVA`
r = []
try:
    r = verificar_tabla_comparativa(TABLA_COMPARATIVA)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('8 — Contraste', r)


## Pregunta 9 — Export LaTeX

### 📘 Subpreguntas
- **9.a** ¿Qué aporta la traza ReAct en el apéndice?


### 🤖 Guía IA — Sección 9: Export LaTeX

**Objetivo:** Implementar export_latex_report y generar informe_sismico.tex para BLDG-B.

**Tu código debe lograr**
- Wrapper que llame generar_latex_report de _verificar
- RUTA_INFORME = Path('informe_sismico.tex')
- Incluir dictamen y traza en el .tex

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `RUTA_INFORME`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- No instalar TeX Live — solo generar .tex
- Escapar caracteres LaTeX via generar_latex_report

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Implementa export_latex_report(...) -> str usando generar_latex_report de _verificar.
Genera informe para BLDG-B con telemetría, RESULTADO_NORMA_B, MSG_ML_B, PROB_RIESGO_B,
TRAZA_HERRAMIENTAS y un dictamen breve. RUTA_INFORME = Path('informe_sismico.tex').
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · RUTA_INFORME

# Tu código debe:
#   1. Definir export_latex_report
#   2. Generar informe_sismico.tex



In [ ]:
# --- Autoevaluación 9 ---
# Requiere (celda «Aquí coloca tu solución»): `RUTA_INFORME`
r = []
try:
    r = verificar_latex_report(RUTA_INFORME, 'BLDG-B')
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('9 — LaTeX', r)


## Pregunta 10 — Puente MCP

### 📘 Subpreguntas
- **10.a** ¿Cómo se mapean tus tools a MCP?


### 🤖 Guía IA — Sección 10: Cierre MCP

**Objetivo:** Reflexionar sobre puente hacia MCP y completar bitácora.

**Tu código debe lograr**
- Definir `EJEMPLO_MCP` = una frase sobre cómo las tools serían un servidor MCP
- Imprimir EJEMPLO_MCP

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `EJEMPLO_MCP`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Las 4 funciones Python = prototipo de servidor MCP
- Completa prompts_entregados.md

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Define EJEMPLO_MCP como string de 1-2 oraciones explicando que get_building_telemetry,
check_seismic_code_compliance, predict_earthquake_vulnerability y export_latex_report
podrían exponerse como tools MCP. Imprime EJEMPLO_MCP.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · EJEMPLO_MCP

# Tu código debe:
#   1. Definir EJEMPLO_MCP
#   2. Imprimir reflexión MCP



In [ ]:
# --- Autoevaluación 10 ---
# Requiere (celda «Aquí coloca tu solución»): `EJEMPLO_MCP`
r = []
try:
    r = [verificar(isinstance(EJEMPLO_MCP, str) and len(EJEMPLO_MCP) > 20, '✅ EJEMPLO_MCP definido.', '❌ Define EJEMPLO_MCP.')]
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('10 — MCP', r)


## Cierre y entrega (vía IA)

### ✍️ Reflexión (3 frases)
- El agente Ollama decidió llamar primero a ___ porque ___.
- La diferencia norma vs ML la observé en ___.
- Para MCP migraría la tool ___ primero.

### 📋 Bitácora
Completa [`prompts_entregados.md`](prompts_entregados.md).

**Checklist:** 10 autoevaluaciones ✅ · `informe_sismico.tex` · Ollama probado · bitácora.
